# Anomaly Detection using Isolation Forest

## Objective

The objective of this notebook is to identify abnormal household electricity consumption patterns using an unsupervised machine learning approach.

Unlike supervised learning, anomaly detection does not require labelled examples of abnormal behaviour. Instead, the algorithm learns the characteristics of normal observations and isolates records that significantly deviate from these patterns.

The final output of this notebook will be an anomaly-labelled dataset that will later be interpreted using SHAP and visualized through the EcoWatt AI dashboard.

## Energy Consumption Anamoly Detection

In [ ]:
#importing libraries
import pandas as pd 
import numpy as np 

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

import matplotlib.pyplot as plt
import matplotlib.dates as mdates


## Dataset Overview

The input dataset contains engineered temporal, statistical, and behavioural features generated during the previous stage of the project.

These features provide richer information than the original measurements and improve the ability of the anomaly detection algorithm to distinguish normal and abnormal energy consumption patterns.

In [ ]:
model_df = pd.read_csv("data/processed/energy_features.csv",parse_dates = ["datetime"])

In [ ]:
print(model_df.shape)
model_df.head()

In [ ]:
model_df.info()

## Feature Selection

Selecting relevant input variables is an important step in machine learning.

Only meaningful numerical features describing household electricity consumption are included in the model training process.

Irrelevant or identifier columns are excluded to reduce noise and improve model performance.

In [ ]:
feature_columns = [

    # Original electrical variables
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    # Temporal
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "year_sin",
    "year_cos",

    # Lag
    "active_power_lag_1",
    "active_power_lag_5",
    "active_power_lag_15",
    "active_power_lag_60",

    # Rolling
    "active_power_rolling_mean_15",
    "active_power_rolling_std_15",
    "active_power_rolling_min_15",
    "active_power_rolling_max_15",

    "active_power_rolling_mean_60",
    "active_power_rolling_std_60",
    "active_power_rolling_min_60",
    "active_power_rolling_max_60",

    # Behaviour
    "active_power_change_1",
    "deviation_from_15min_mean",
    "deviation_from_60min_mean",

    # Log transformed
    "active_power_change_rate_log",
    "rolling_zscore_15_log",
    "rolling_zscore_60_log"
]

In [ ]:
X = model_df[feature_columns].copy()
print(X.shape)

## Why Feature Scaling?

The engineered features have different numerical ranges.

For example, voltage values are considerably larger than sub-metering measurements.

Standardization transforms each feature to a comparable scale, ensuring that no single feature dominates the anomaly detection process due to its magnitude.

In [ ]:
sc=StandardScaler()
X_scaled = sc.fit_transform(X)
X_scaled.shape

## Isolation Forest Algorithm

Isolation Forest is an unsupervised anomaly detection algorithm specifically designed to identify rare and unusual observations.

Instead of profiling normal behaviour, the algorithm isolates individual data points using randomly generated decision trees.

Observations requiring fewer partitions to isolate are more likely to be anomalous.

In [ ]:
iso_model = IsolationForest(n_estimators=300,contamination=0.01,max_samples="auto",random_state=42,n_jobs=-1)

## Model Training

During training, the Isolation Forest constructs multiple random isolation trees.

Each tree recursively partitions the feature space until individual observations become isolated.

The average path length across all trees is then used to estimate the degree of abnormality.

In [ ]:
iso_model.fit(X_scaled)

## Model Persistence

The trained Isolation Forest model and preprocessing objects are saved for future use.

Saving the model eliminates the need for retraining and allows consistent predictions during dashboard deployment.

In [ ]:
import joblib

joblib.dump(sc,"models/standard_scaler.pkl")
joblib.dump(iso_model,"models/isolation_forest.pkl")

## Anomaly Prediction

After training, the model assigns each observation one of two labels:

- **1** → Normal Observation
- **-1** → Anomalous Observation

These predictions are later combined with anomaly scores for further interpretation.

In [ ]:
model_df["anomaly"] = iso_model.predict(X_scaled)
model_df["anomaly"].value_counts()

In [ ]:
model_df["anomaly"] =(model_df["anomaly"] == -1).astype(int)
print(model_df["anomaly"].value_counts())
anomaly_rate = (model_df["anomaly"].mean()*100)
print(f"Anomaly Rate:{anomaly_rate:.2f}%")
print(model_df["anomaly"].sum())

## Anomaly Score

Besides predicting anomaly labels, Isolation Forest computes a continuous anomaly score.

Lower scores indicate observations that are more likely to represent abnormal energy consumption patterns.

These scores provide additional insight into the severity of detected anomalies.

In [ ]:
model_df["anomaly_score"] = iso_model.decision_function(X_scaled)
model_df["anomaly_score"].describe()

In [ ]:
top_anomalies = model_df.sort_values(
    "anomaly_score"
).head(20)

top_anomalies[
    [
        "datetime",
        "segment_id",
        "Global_active_power",
        "anomaly_score",
        "active_power_change_rate_log",
        "rolling_zscore_15_log",
        "rolling_zscore_60_log"
    ]
]

## Visualization

Visualizing detected anomalies helps validate model behaviour.

Scatter plots and time-series visualizations make it easier to identify abnormal consumption events and understand when they occurred.

## Time Series Analysis

In [ ]:
plt.figure(figsize=(18,6))

plt.plot(
    model_df["datetime"],
    model_df["Global_active_power"],
    linewidth=0.5,
    label="Power"
)

plt.scatter(
    model_df.loc[model_df["anomaly"]==1,"datetime"],
    model_df.loc[model_df["anomaly"]==1,"Global_active_power"],
    color="red",
    s=6,
    label="Anomaly"
)

plt.title("Detected Energy Consumption Anomalies")

plt.xlabel("Date")

plt.ylabel("Global Active Power (kW)")

plt.legend()

plt.tight_layout()

plt.show()

## Monthly Distribution of Anomalies

In [ ]:
model_df["year_month"] = (
    model_df["datetime"]
    .dt.to_period("M")
    .astype(str)
)

monthly_anomalies = (
    model_df.groupby("year_month")["anomaly"]
    .sum()
)

In [ ]:
plt.figure(figsize=(18,5))

monthly_anomalies.plot()

plt.title("Monthly Number of Detected Anomalies")

plt.ylabel("Anomaly Count")

plt.xticks(rotation=90)

plt.tight_layout()

plt.show()

## Hour of Day Analysis

In [ ]:
hourly = (model_df.groupby("hour")["anomaly"].sum())

In [ ]:
plt.figure(figsize=(10,8))

hourly.plot(kind="bar")

plt.title("Hourly Distribution of Anomalies")

plt.xlabel("Hour")

plt.ylabel("Detected Anomalies")

plt.tight_layout()

plt.show()

In [ ]:
comparison = (
    model_df
    .groupby("anomaly")
    [
        [
            "Global_active_power",
            "Voltage",
            "Global_intensity",
            "active_power_change_rate_log",
            "rolling_zscore_15_log",
            "rolling_zscore_60_log"
        ]
    ]
    .mean()
)

comparison

In [ ]:
detected_anomalies = (
    model_df[
        model_df["anomaly"]==1
    ]
)

print(detected_anomalies.shape)

## Generated Outputs

This notebook produces:

- Trained Isolation Forest model
- StandardScaler object
- Anomaly labels
- Anomaly scores
- Processed dataset for dashboard visualization

These outputs will be reused throughout the remaining stages of the EcoWatt AI project.

In [ ]:
model_df.to_csv("data/processed/energy_with_anomalies.csv",index=False)

In [ ]:
detected_anomalies.to_csv("data/processed/detected_anomalies.csv",index=False)

In [ ]:
def explain_anomaly(row):

    reasons = []

    if row["rolling_zscore_15_log"] > 2:
        reasons.append("Large short-term consumption deviation")

    if row["rolling_zscore_60_log"] > 2:
        reasons.append("Large hourly consumption deviation")

    if row["active_power_change_rate_log"] > 2:
        reasons.append("Sudden increase in energy usage")

    if row["Voltage"] < 220:
        reasons.append("Lower than normal voltage")

    if row["Voltage"] > 245:
        reasons.append("Higher than normal voltage")

    if len(reasons)==0:
        reasons.append("Complex multivariate anomaly")

    return "; ".join(reasons)

In [ ]:
detected_anomalies["reason"] = (
    detected_anomalies.apply(
        explain_anomaly,
        axis=1
    )
)

In [ ]:
detected_anomalies[
    [
        "datetime",
        "Global_active_power",
        "anomaly_score",
        "reason"
    ]
].head(20)

In [ ]:
print(model_df["anomaly"].value_counts())

# Conclusion

The anomaly detection pipeline successfully identified unusual household energy consumption patterns using the Isolation Forest algorithm.

Major accomplishments of this notebook include:

- Selection of informative engineered features
- Feature standardization using StandardScaler
- Training of the Isolation Forest model
- Prediction of anomaly labels
- Computation of anomaly scores
- Visualization of abnormal observations
- Export of the trained model and processed dataset

The generated anomaly predictions provide the foundation for explainable AI using SHAP and enable real-time visualization through the EcoWatt AI dashboard.